In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import json
import time
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

from google.colab import drive
drive.mount("/content/drive/")

PROJECT_DIR = Path("/content/drive/MyDrive/Underwater-Image-Data-set-main")
V1_DIR = PROJECT_DIR / "Dataset_V1"
TUNING_DIR = V1_DIR / "Classical_Tuning"
TUNING_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_FILE = V1_DIR / "Dataset_V1_splits.csv"
BASELINE_DIR = V1_DIR / "Classical_Baselines"

df = pd.read_csv(SPLIT_FILE)

val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

report_file = BASELINE_DIR / "Dataset_V1_classical_baseline_report.json"

selected_method = None

if report_file.exists():
    with open(report_file, "r") as f:
        report = json.load(f)

    selected_method = (
        report.get("selected_method")
        or report.get("best_method")
    )

if selected_method is None:

    comparison_file = BASELINE_DIR / "baseline_comparison.csv"
    comparison = pd.read_csv(comparison_file)

    print("Columns found in baseline_comparison.csv:")
    print(list(comparison.columns))

    method_col = None

    for col in ["Method", "method", "METHOD"]:
        if col in comparison.columns:
            method_col = col
            break

    if method_col is None:
        raise ValueError(
            "Could not find the method column in baseline_comparison.csv"
        )

    psnr_col = next(
        (c for c in comparison.columns if c.lower() == "psnr"),
        None
    )

    ssim_col = next(
        (c for c in comparison.columns if c.lower() == "ssim"),
        None
    )

    edge_col = next(
        (
            c for c in comparison.columns
            if "edge" in c.lower()
        ),
        None
    )

    rank_columns = []

    if psnr_col:
        comparison["PSNR_rank"] = comparison[psnr_col].rank(
            ascending=False
        )
        rank_columns.append("PSNR_rank")

    if ssim_col:
        comparison["SSIM_rank"] = comparison[ssim_col].rank(
            ascending=False
        )
        rank_columns.append("SSIM_rank")

    if edge_col:
        comparison["Edge_rank"] = comparison[edge_col].rank(
            ascending=False
        )
        rank_columns.append("Edge_rank")

    if not rank_columns:
        raise ValueError(
            "No usable PSNR, SSIM or edge metric found."
        )

    comparison["Average_Rank"] = comparison[rank_columns].mean(axis=1)

    selected_method = comparison.loc[
        comparison["Average_Rank"].idxmin(),
        method_col
    ]

print("Shortlisted method:", selected_method)
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))
print("Test set used for tuning: NO")


def load_rgb(path):
    img = cv2.imread(str(path))

    if img is None:
        raise ValueError(f"Could not read image: {path}")

    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def clahe_enhance(img, clip_limit=2.0, tile_grid=(8, 8)):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)

    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=float(clip_limit),
        tileGridSize=tuple(tile_grid)
    )

    l = clahe.apply(l)

    return cv2.cvtColor(
        cv2.merge([l, a, b]),
        cv2.COLOR_LAB2RGB
    )


def gamma_enhance(img, gamma=1.3):
    img_float = img.astype(np.float32) / 255.0

    corrected = np.power(
        img_float,
        float(gamma)
    )

    return np.clip(
        corrected * 255,
        0,
        255
    ).astype(np.uint8)


def gray_world(img, strength=1.0):
    img_float = img.astype(np.float32)

    means = img_float.reshape(-1, 3).mean(axis=0)

    gray = means.mean()

    scale = gray / (means + 1e-6)

    corrected = img_float * scale

    corrected = np.clip(
        corrected,
        0,
        255
    )

    result = (
        strength * corrected
        + (1 - strength) * img_float
    )

    return np.clip(
        result,
        0,
        255
    ).astype(np.uint8)


def edge_preservation(enhanced, target):

    enhanced_gray = cv2.cvtColor(
        enhanced,
        cv2.COLOR_RGB2GRAY
    )

    target_gray = cv2.cvtColor(
        target,
        cv2.COLOR_RGB2GRAY
    )

    enhanced_edges = cv2.Canny(
        enhanced_gray,
        100,
        200
    )

    target_edges = cv2.Canny(
        target_gray,
        100,
        200
    )

    target_count = np.sum(
        target_edges > 0
    )

    if target_count == 0:
        return 0.0

    overlap = np.logical_and(
        enhanced_edges > 0,
        target_edges > 0
    )

    return float(
        np.sum(overlap) / target_count
    )


def brightness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_RGB2GRAY
    )

    return float(
        np.mean(gray)
    )


def contrast(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_RGB2GRAY
    )

    return float(
        np.std(gray)
    )


def edge_count(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_RGB2GRAY
    )

    return int(
        np.sum(
            cv2.Canny(
                gray,
                100,
                200
            ) > 0
        )
    )


def apply_method(img, method, params):

    method_lower = (
        method
        .lower()
        .replace("-", "")
        .replace("_", "")
    )

    if method_lower == "clahe":

        return clahe_enhance(
            img,
            params["clip_limit"],
            tuple(params["tile_grid"])
        )

    elif method_lower == "gamma":

        return gamma_enhance(
            img,
            params["gamma"]
        )

    elif method_lower in [
        "grayworld",
        "grayworldwhitebalance"
    ]:

        return gray_world(
            img,
            params["strength"]
        )

    else:

        raise ValueError(
            f"Unsupported method: {method}"
        )


method_clean = (
    selected_method
    .lower()
    .replace("-", "")
    .replace("_", "")
    .replace(" ", "")
)


if method_clean == "clahe":

    search_space = [
        {
            "clip_limit": x,
            "tile_grid": [8, 8]
        }
        for x in [1.0, 2.0, 3.0, 4.0]
    ]

    search_space += [
        {
            "clip_limit": 2.0,
            "tile_grid": [4, 4]
        },
        {
            "clip_limit": 2.0,
            "tile_grid": [16, 16]
        }
    ]


elif method_clean == "gamma":

    search_space = [
        {
            "gamma": x
        }
        for x in [
            0.8,
            1.0,
            1.2,
            1.3,
            1.5,
            1.8
        ]
    ]


elif method_clean in [
    "grayworld",
    "grayworldwhitebalance"
]:

    search_space = [
        {
            "strength": x
        }
        for x in [
            0.5,
            0.75,
            1.0
        ]
    ]


else:

    raise ValueError(
        f"Unknown shortlisted method: {selected_method}"
    )


results = []


for config_id, params in enumerate(
    search_space,
    start=1
):

    psnr_values = []
    ssim_values = []
    edge_values = []

    brightness_changes = []
    contrast_changes = []
    edge_changes = []

    times = []

    for _, row in val_df.iterrows():

        input_img = load_rgb(
            PROJECT_DIR / row["input"]
        )

        target_img = load_rgb(
            PROJECT_DIR / row["target"]
        )

        start = time.perf_counter()

        enhanced = apply_method(
            input_img,
            selected_method,
            params
        )

        elapsed = (
            time.perf_counter()
            - start
        )

        psnr_values.append(
            peak_signal_noise_ratio(
                target_img,
                enhanced,
                data_range=255
            )
        )

        ssim_values.append(
            structural_similarity(
                target_img,
                enhanced,
                channel_axis=2,
                data_range=255
            )
        )

        edge_values.append(
            edge_preservation(
                enhanced,
                target_img
            )
        )

        brightness_changes.append(
            abs(
                brightness(enhanced)
                - brightness(input_img)
            )
        )

        contrast_changes.append(
            abs(
                contrast(enhanced)
                - contrast(input_img)
            )
        )

        edge_changes.append(
            abs(
                edge_count(enhanced)
                - edge_count(input_img)
            )
        )

        times.append(elapsed)

    results.append({

        "config_id": config_id,

        "method": selected_method,

        "parameters": json.dumps(
            params
        ),

        "PSNR": np.mean(
            psnr_values
        ),

        "SSIM": np.mean(
            ssim_values
        ),

        "Edge Preservation": np.mean(
            edge_values
        ),

        "Mean Brightness Change": np.mean(
            brightness_changes
        ),

        "Mean Contrast Change": np.mean(
            contrast_changes
        ),

        "Mean Edge Count Change": np.mean(
            edge_changes
        ),

        "Mean Processing Time": np.mean(
            times
        )
    })


results_df = pd.DataFrame(
    results
)


results_df["PSNR_rank"] = results_df[
    "PSNR"
].rank(
    ascending=False
)

results_df["SSIM_rank"] = results_df[
    "SSIM"
].rank(
    ascending=False
)

results_df["Edge_rank"] = results_df[
    "Edge Preservation"
].rank(
    ascending=False
)


results_df["Average_Rank"] = results_df[
    [
        "PSNR_rank",
        "SSIM_rank",
        "Edge_rank"
    ]
].mean(axis=1)


results_df = results_df.sort_values(
    "Average_Rank"
).reset_index(drop=True)


results_df["selected"] = False

results_df.loc[
    0,
    "selected"
] = True


results_file = (
    TUNING_DIR /
    "parameter_tuning_results.csv"
)

results_df.to_csv(
    results_file,
    index=False
)


best = results_df.iloc[0]


best_config = {

    "selected_method":
        selected_method,

    "best_parameters":
        json.loads(
            best["parameters"]
        ),

    "validation_PSNR":
        float(best["PSNR"]),

    "validation_SSIM":
        float(best["SSIM"]),

    "validation_Edge_Preservation":
        float(
            best["Edge Preservation"]
        ),

    "selection_rule":
        "Best average rank across PSNR, SSIM and Edge Preservation on validation set",

    "validation_samples":
        int(len(val_df)),

    "test_samples":
        int(len(test_df)),

    "test_used_for_selection":
        False
}


best_file = (
    TUNING_DIR /
    "best_validated_method.json"
)

with open(
    best_file,
    "w"
) as f:

    json.dump(
        best_config,
        f,
        indent=4
    )


artifact_results = []

for _, row in results_df.iterrows():

    artifact_results.append({

        "config_id":
            row["config_id"],

        "method":
            row["method"],

        "parameters":
            row["parameters"],

        "brightness_change":
            row["Mean Brightness Change"],

        "contrast_change":
            row["Mean Contrast Change"],

        "edge_count_change":
            row["Mean Edge Count Change"],

        "potential_brightness_artifact":
            row["Mean Brightness Change"] > 40,

        "potential_contrast_artifact":
            row["Mean Contrast Change"] > 40,

        "potential_edge_artifact":
            row["Mean Edge Count Change"] > 1500
    })


artifact_df = pd.DataFrame(
    artifact_results
)


artifact_df.to_csv(
    TUNING_DIR /
    "artifact_check.csv",
    index=False
)


print("\n" + "=" * 60)
print("PARAMETER TUNING COMPLETE")
print("=" * 60)

print(
    "\nShortlisted method:",
    selected_method
)

print(
    "Configurations tested:",
    len(search_space)
)

print(
    "\nBEST VALIDATED CONFIGURATION"
)

print(
    "Parameters:",
    best["parameters"]
)

print(
    "Validation PSNR:",
    round(
        best["PSNR"],
        4
    )
)

print(
    "Validation SSIM:",
    round(
        best["SSIM"],
        4
    )
)

print(
    "Validation Edge Preservation:",
    round(
        best["Edge Preservation"],
        4
    )
)

print("\nSaved files:")
print(results_file)
print(TUNING_DIR / "artifact_check.csv")
print(best_file)

print(
    "\nFinal test set used for tuning: NO"
)

print(
    "The test set remains untouched."
)

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
Columns found in baseline_comparison.csv:
['method', 'PSNR', 'SSIM', 'Edge_Preservation']
Shortlisted method: Gamma
Validation samples: 47
Test samples: 48
Test set used for tuning: NO

PARAMETER TUNING COMPLETE

Shortlisted method: Gamma
Configurations tested: 6

BEST VALIDATED CONFIGURATION
Parameters: {"gamma": 0.8}
Validation PSNR: 13.6977
Validation SSIM: 0.6623
Validation Edge Preservation: 0.0058

Saved files:
/content/drive/MyDrive/Underwater-Image-Data-set-main/Dataset_V1/Classical_Tuning/parameter_tuning_results.csv
/content/drive/MyDrive/Underwater-Image-Data-set-main/Dataset_V1/Classical_Tuning/artifact_check.csv
/content/drive/MyDrive/Underwater-Image-Data-set-main/Dataset_V1/Classical_Tuning/best_validated_method.json

Final test set used for tuning: NO
The test set remains untouched.
